In [ ]:
import time, random, sys, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# OFFICIAL PHYSIONET CHALLENGE 2025 SCORER
def compute_challenge_score(labels, outputs, fraction_capacity=0.05, num_permutations=10000, seed=12345):
    """OFFICIAL PhysioNet Challenge 2025 top5-TPR scorer"""
    assert len(labels) == len(outputs), "Labels and outputs must have same length"
    num_instances = len(labels)
    capacity = int(fraction_capacity * num_instances)  # Top 5%
    
    labels = np.asarray(labels).astype(np.float64)
    outputs = np.asarray(outputs).astype(np.float64)
    
    np.random.seed(seed)
    tp = np.zeros(num_permutations)
    
    for i in range(num_permutations):
        permuted_idx = np.random.permutation(num_instances)
        permuted_labels = labels[permuted_idx]
        permuted_outputs = outputs[permuted_idx]
        ordered_idx = np.argsort(permuted_outputs)[::-1]
        ordered_labels = permuted_labels[ordered_idx]
        tp[i] = np.sum(ordered_labels[:capacity] == 1)
    
    tp_mean = np.mean(tp)
    fn = np.sum(labels == 1) - tp_mean
    return tp_mean / (tp_mean + fn + 1e-8)

print(" OFFICIAL PhysioNet Challenge Scorer LOADED")

# SEED everything
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")

# PROJECT STRUCTURE
from pathlib import Path
from datetime import datetime

def find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "data").exists(): 
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "processed"
sys.path.append(str(PROJECT_ROOT))

# EXPERIMENT TRACKING
EXP_NAME = "vit_contour_fixed_corrected_20260201"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EXP_DIR = PROJECT_ROOT / "experiments" / EXP_NAME / RUN_ID
EXP_DIR.mkdir(parents=True, exist_ok=True)

print(f" PROJECT_ROOT: {PROJECT_ROOT}")
print(f" DATA_DIR:     {DATA_DIR}")
print(f" EXP_DIR:      {EXP_DIR}")
print(" Cell 1 Complete - Ready for training!")


In [ ]:
cell_start = time.time()

datasets = ["ptbxl", "sami_trop", "code15"]
dfs = []

for ds in datasets:
    csv_path = DATA_DIR / "metadata" / f"{ds}_metadata.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing: {csv_path}")
    
    df = pd.read_csv(csv_path)
    df['dataset'] = ds
    df['label'] = df['label'].astype(float)
    
    if ds == 'code15':
        df.loc[df['label'] > 0.5, 'label'] = 0.8  # Soft labels
        df.loc[df['label'] <= 0.5, 'label'] = 0.2
    
    dfs.append(df)
    print(f"Loaded {ds}: {len(df):,} records")

df_all = pd.concat(dfs, ignore_index=True)

def img_exists(p):
    full_path = (PROJECT_ROOT / Path(str(p))).resolve()
    return full_path.exists()

exists_mask = df_all['img_path'].apply(img_exists)
if (~exists_mask).sum() > 0:
    print(f"Dropped {(~exists_mask).sum():,} missing images")
    df_all = df_all[exists_mask].reset_index(drop=True)

df_all['label_bin'] = (df_all['label'] > 0.5).astype(int)
df_all['strat_key'] = df_all['dataset'] + '_' + df_all['label_bin'].astype(str)

train_df, temp_df = train_test_split(df_all, test_size=0.2, stratify=df_all['strat_key'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['strat_key'], random_state=SEED)

print(f"Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")
print(f"✓ Cell 2: {time.time()-cell_start:.1f}s")


In [ ]:
cell_start = time.time()

USE_SAMPLE = True  # Full training → False
SAMPLE_FRAC = 0.1

if USE_SAMPLE:
    print(f"SAMPLE MODE: {SAMPLE_FRAC*100:.0f}% data")
    for split, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        sampled = df.groupby('strat_key', group_keys=False).apply(
            lambda x: x.sample(frac=SAMPLE_FRAC, random_state=SEED)
        ).reset_index(drop=True)
        locals()[split.lower() + '_df'] = sampled
        print(f"{split}: {len(sampled):4,} ({sampled['label_bin'].sum():3,} pos)")

print(f"Positives - Train: {train_df['label_bin'].sum():,} | Val: {val_df['label_bin'].sum():,} | Test: {test_df['label_bin'].sum():,}")
print(f"✓ Cell 3: {time.time()-cell_start:.1f}s")


In [ ]:
cell_start = time.time()

class ECGImageDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.img_paths = [PROJECT_ROOT / Path(p) for p in self.df['img_path']]
        self.labels = self.df['label'].values.astype(np.float32)  # Soft labels
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        img = np.load(self.img_paths[idx], allow_pickle=False).astype(np.float32)
        assert img.shape == (3, 24, 2048), f"Shape {img.shape}"
        
        # Normalize images
        if img.max() > 4:
            img = (img - 128) / 128
        else:
            img = np.clip(img, -3, 3) / 3
            
        return torch.from_numpy(img), torch.tensor(self.labels[idx])

train_ds = ECGImageDataset(train_df)
val_ds = ECGImageDataset(val_df)
test_ds = ECGImageDataset(test_df)

pos_count = (train_df['label'] > 0.5).sum()
neg_count = (train_df['label'] <= 0.5).sum()
pos_weight = neg_count / pos_count
print(f"Pos weight: {pos_weight:.1f}x")

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

print(f"✓ Cell 4: {time.time()-cell_start:.1f}s")


In [ ]:
import time
import torch
import torch.nn as nn

cell_start = time.time()

class ViTClassifier(nn.Module):
    def __init__(self, embed_dim=384, heads=6, mlp_ratio=4, dropout=0.1, depth=8):
        super().__init__()
        
        # Patch embedding: 24x2048 -> 3x128 patches (stride 8x16)
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=(8,16), stride=(8,16), bias=False)
        num_patches = (24//8) * (2048//16)  # 3 * 128 = 384 patches
        
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim) * 0.02)
        
        # Transformer blocks (norm_first = True for stability)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                embed_dim, heads, int(embed_dim * mlp_ratio), dropout,
                activation='gelu', batch_first=True, norm_first=True
            ) for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(embed_dim, 1)
        )
    
    def forward(self, x):
        B = x.shape[0]
        
        # Patch embedding + flatten
        x = self.patch_embed(x).flatten(2).transpose(1, 2)  # B, N, D
        
        # Add CLS token + positional embedding
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1) + self.pos_embed
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # STANDARD ViT: Final CLS token through norm + head
        x = self.norm(x[:, 0])
        return self.head(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ViTClassifier().to(device)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"ViT-{total_params:.1f}M params on {device}")

# Test forward pass
with torch.no_grad():
    x = torch.randn(2, 3, 24, 2048).to(device)
    assert model(x).shape == (2,)
    print("✓ Forward pass OK")

print(f"✓ Cell 5: {time.time()-cell_start:.1f}s")

In [ ]:
cell_start = time.time()

EPOCHS = 5
LR = 2e-4
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR/20)

history = {'epoch':[], 'loss':[], 'auc':[], 'auprc':[], 'f1':[], 'challenge':[]}
best_challenge = 0
best_path = EXP_DIR / "best_model.pth"

print(f"\n TRAINING: {EPOCHS} epochs | LR={LR} | PosWeight={pos_weight:.1f}x")

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        logits = model(imgs).unsqueeze(1)  #  FIXED: [B] -> [B,1]
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validation (OFFICIAL BINARY LABELS FOR SCORING)
    model.eval()
    preds, trues_bin = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            probs = torch.sigmoid(model(imgs)).cpu().numpy().flatten()
            preds.extend(probs)
            trues_bin.extend((labels.cpu().numpy() > 0.5).astype(int))
    
    preds = np.array(preds)
    trues_bin = np.array(trues_bin)
    
    # OFFICIAL METRICS
    auc = roc_auc_score(trues_bin, preds)
    auprc = average_precision_score(trues_bin, preds)
    f1 = f1_score(trues_bin, preds >= 0.5)
    challenge = compute_challenge_score(trues_bin, preds)  # BINARY labels!
    
    history['epoch'].append(epoch+1)
    history['loss'].append(train_loss/len(train_loader))
    history['auc'].append(auc)
    history['auprc'].append(auprc)
    history['f1'].append(f1)
    history['challenge'].append(challenge)
    
    if challenge > best_challenge:
        best_challenge = challenge
        torch.save(model.state_dict(), best_path)
    
    print(f"Epoch {epoch+1:02d} | Loss:{train_loss/len(train_loader):.4f} | "
          f"AUC:{auc:.4f} | AUPRC:{auprc:.4f} | F1:{f1:.3f} | Challenge:{challenge:.4f} "
          f"{'BEST!' if challenge>best_challenge else ''}")

pd.DataFrame(history).to_csv(EXP_DIR/'training_history.csv', index=False)
print(f"\n Training done! Best Challenge: {best_challenge:.4f}")
print(f"Model saved: {best_path}")
print(f"✓ Cell 6: {time.time()-cell_start:.1f}s")


In [ ]:
# Load best model and evaluate on test set
model.load_state_dict(torch.load(best_path))
model.eval()

test_preds, test_trues_bin = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        probs = torch.sigmoid(model(imgs)).cpu().numpy().flatten()
        test_preds.extend(probs)
        test_trues_bin.extend((labels.cpu().numpy() > 0.5).astype(int))

test_preds = np.array(test_preds)
test_trues_bin = np.array(test_trues_bin)

test_auc = roc_auc_score(test_trues_bin, test_preds)
test_auprc = average_precision_score(test_trues_bin, test_preds)
test_f1 = f1_score(test_trues_bin, test_preds >= 0.5)
test_challenge = compute_challenge_score(test_trues_bin, test_preds)

print(f"\n FINAL TEST RESULTS")
print(f"AUC:     {test_auc:.4f}")
print(f"AUPRC:   {test_auprc:.4f}")
print(f"F1:      {test_f1:.3f}")
print(f"Challenge: {test_challenge:.4f}")
print(f"Expected LB: ~0.40-0.50")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
df_hist = pd.DataFrame(history)

axes[0,0].plot(df_hist['epoch'], df_hist['loss'], 'b-', linewidth=2)
axes[0,0].set_title('Training Loss')
axes[0,0].set_xlabel('Epoch')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(df_hist['epoch'], df_hist['auc'], 'g-', linewidth=2)
axes[0,1].set_title(f'Validation AUC (Max: {df_hist["auc"].max():.3f})')
axes[0,1].set_xlabel('Epoch')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(df_hist['epoch'], df_hist['challenge'], 'r-', linewidth=2)
axes[1,0].axhline(y=best_challenge, color='r', linestyle='--', linewidth=2, 
                  label=f'Best: {best_challenge:.4f}')
axes[1,0].set_title('Challenge Score (Primary Metric)')
axes[1,0].set_xlabel('Epoch')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(df_hist['epoch'], df_hist['f1'], 'purple', linewidth=2)
axes[1,1].set_title(f'Validation F1 (Max: {df_hist["f1"].max():.3f})')
axes[1,1].set_xlabel('Epoch')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle(f'ViT Training Progress | Best Challenge: {best_challenge:.4f}', fontsize=14)
plt.tight_layout()
plt.savefig(EXP_DIR / 'training_curves.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
print("DATASET BREAKDOWN")
summary_df = pd.DataFrame({
    'Split': ['Train', 'Val', 'Test', 'Overall'],
    'N': [len(train_df), len(val_df), len(test_df), len(df_all)],
    'Positives': [train_df['label_bin'].sum(), val_df['label_bin'].sum(), 
                  test_df['label_bin'].sum(), df_all['dataset'].value_counts().sum()],
    'Prevalence': [f"{train_df['label_bin'].mean():.2%}", f"{val_df['label_bin'].mean():.2%}", 
                   f"{test_df['label_bin'].mean():.2%}", f"{df_all['label_bin'].mean():.2%}"]
})
print(summary_df.round(2))

print("\n Dataset Distribution:")
print(df_all['dataset'].value_counts())
print("\n Dataset Prevalence:")
print(pd.crosstab(df_all['dataset'], df_all['label_bin'], 
                  normalize='index').round(3)*100, "← % Positive")


In [ ]:
total_params = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

print(f" ViT Model Summary")
print(f"Total params:     {total_params:.1f}M")
print(f"Trainable params: {trainable_params:.1f}M ({trainable_params/total_params*100:.0f}%)")
print(f"Best Checkpoint:  {best_path}")
print(f"Device:           {device}")
print(f"Best Challenge:   {best_challenge:.4f}")

# Skip gradient check (only available during training)
print("\n Model ready for inference & ensemble!")


In [ ]:
## CELL 11.5: CONFUSION MATRIX + KEY METRICS
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Optimal threshold for F1 (Challenge uses top-5%, but CM needs binary)
from sklearn.metrics import f1_score
thresh_f1 = np.linspace(0.1, 0.9, 81)
f1_scores = [f1_score(test_trues_bin, test_preds >= t) for t in thresh_f1]
best_thresh = thresh_f1[np.argmax(f1_scores)]
test_preds_bin = (test_preds >= best_thresh).astype(int)

# Confusion Matrix
cm = confusion_matrix(test_trues_bin, test_preds_bin)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title('Confusion Matrix (Raw Counts)')
ax1.set_xlabel('Predicted'); ax1.set_ylabel('Actual')

# Normalized percentages
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues', ax=ax2)
ax2.set_title(f'Confusion Matrix (Normalized @ F1 thresh={best_thresh:.3f})')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(EXP_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Print metrics
tn, fp, fn, tp = cm.ravel()
print(f" CONFUSION MATRIX METRICS @ F1 Optimal Threshold ({best_thresh:.3f})")
print(f"TP: {tp:>6,} | TN: {tn:>6,} | FP: {fp:>6,} | FN: {fn:>6,}")
print(f"Precision:  {tp/(tp+fp):.3f} | Recall:    {tp/(tp+fn):.3f}")
print(f"Specificity:{tn/(tn+fp):.3f} | F1:        {f1_score(test_trues_bin, test_preds_bin):.3f}")


In [ ]:
top_k = int(0.05 * len(test_preds))
top_indices = np.argsort(test_preds)[-top_k:]
top_labels = test_trues_bin[top_indices]
recall_at_5 = top_labels.mean()

print(f"\n TOP 5% PERFORMANCE (Challenge Metric)")
print(f"Test Size:        {len(test_preds):>6,} records")
print(f"Top 5%:           {top_k:>6,} records")
print(f"Positives found:  {top_labels.sum():>6,} ({recall_at_5:.1%})")
print(f"Total Positives:  {test_trues_bin.sum():>6,} ({test_trues_bin.mean():.1%})")
print(f"Challenge Score:  {test_challenge:.4f} ")

# Quick calibration plot
plt.figure(figsize=(8, 6))
bins = np.linspace(0, 1, 11)
bin_indices = np.digitize(test_preds, bins) - 1
bin_props = np.array([test_trues_bin[bin_indices == i].mean() for i in range(10)])
bin_confs = np.array([test_preds[bin_indices == i].mean() for i in range(10)])

plt.plot(bin_confs, bin_props, 'o-', linewidth=3, markersize=8, label='Model')
plt.plot([0,1], [0,1], 'r--', linewidth=2, label='Perfect')
plt.xlabel('Predicted Probability', fontsize=12)
plt.ylabel('Actual Fraction Positive', fontsize=12)
plt.title(f'Calibration Plot | Challenge: {test_challenge:.3f}', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.savefig(EXP_DIR / 'calibration.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Core outputs for ensemble/submission
np.save(EXP_DIR / 'test_predictions.npy', test_preds)
np.save(EXP_DIR / 'test_labels.npy', test_trues_bin)

submission_df = pd.DataFrame({
    'ecg_id': range(len(test_preds)),
    'probability': test_preds.clip(0,1)  # Ensure valid probabilities
})
submission_df.to_csv(EXP_DIR / 'test_predictions.csv', index=False)

# Final performance table
perf_table = pd.DataFrame({
    'Metric': ['Challenge Score', 'AUC', 'AUPRC', 'F1@0.5', 'Top5% Recall'],
    'Test': [f"{test_challenge:.4f}", f"{test_auc:.4f}", f"{test_auprc:.4f}", 
             f"{test_f1:.3f}", f"{recall_at_5:.1%}"]
})
perf_table.to_csv(EXP_DIR / 'performance_summary.csv', index=False)

print(f"\n PIPELINE COMPLETE! {EXP_DIR}")
print(f" Best Challenge: {best_challenge:.4f}")
print(f" Test Challenge: {test_challenge:.4f}")
print(f" Top5% Recall:   {recall_at_5:.1%}")
print(f"\n Competition Files:")
print(f"   {submission_df.shape[0]:,} predictions → test_predictions.csv")
print(f"   Model weights   → best_model.pth") 
print(f"   Full history    → training_history.csv")
print(f"   Plots           → *.png")
print(f"\n READY FOR ENSEMBLE & SUBMISSION! ")
